# Arquitetura Transformer

## 1 Distribuição de parâmetros do modelo GPT
Na última aula você aprendeu como construir a estrutura do transformer para criar o modelo GPT por completo. Um transformer é basicamente composto por um módulo de atenção e um módulo feed forward. Já o modelo GPT é composto pela camada de embeddings, seguido de uma pilha de transformers, com uma camada linear de output no final. À medida que vamos aumentando as dimensões das matrizes e empilhando um número maior de transformers, a capacidade do modelo aumenta, mas a quantidade de parâmetros cresce consideravelmente, chegando na cada dos bilhões, ou até mesmo trilhões, de parâmetros. Daí vem o nome que conhecemos: Large Languange Models.

Vamos verificar como estes pesos se distribuem ao longo da estrutura do modelo GPT para termos uma ideia da quantidade de pesos usada nos diferentes componentes do modelo. Para isso, recrie o modelo GPT visto na última aula, calcule e compare o número de parâmetros contidos na camada de embedding, nas camadas feed forward dos transformers, nos módulos de atenção dos transformers e a quantidade de parâmetros na camada de output geracional.

In [67]:
import torch
import torch.nn as nn

# DEFINA AS CLASSES PARA RECRIAR O MODELO GPTModel

# da ultima aula
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec



class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))




class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)




class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift



class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x




class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential( # vai fazer esse trem de transformer 12 vezes no GPT2
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


In [68]:
# APENAS EXECUTE ESTA CÉLULA

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

model = GPTModel(GPT_CONFIG_124M)

In [69]:
emb_params_count = 0
att_params_count = 0
ff_params_count = 0
output_params_count = 0


# FAÇA A CONTAGEM DOS PARÂMETROS USANDO AS VARIÁVEIS DEFINIDAS ACIMA

# Só usar a lógica padrão de contar parametros no torch com .parameter() e .numel() que nem o exemplo do livro mostra.


# deixei na forma de loop for para ficar mais facil deu visualizar numa futura revisão

for parametro in model.tok_emb.parameters():
    emb_params_count += parametro.numel()
for parametro in model.pos_emb.parameters():
    emb_params_count += parametro.numel()




# emb_params_count = sum(p.numel() for p in model.tok_emb.parameters()) + sum(p.numel() for p in model.pos_emb.parameters()) # como embedding vou considerar os de posição tbm, logo tok_emb + pos_emb

# Isso daqui pegaria de todo o transformer e não do att exclusivamente, hmm...
# att_params_count = sum(p.numel() for p in model.trf_blocks.parameters())


# att_params_count = sum(p.numel() for block in model.trf_blocks for p in block.att.parameters())

for bloco in model.trf_blocks:
    for parametro in bloco.att.parameters():
        att_params_count += parametro.numel()

for bloco in model.trf_blocks:
    for parametro in bloco.ff.parameters():
        ff_params_count += parametro.numel()






# ff_params_count = sum(p.numel() for p in model.trf_blocks.parameters())
# ff_params_count = sum(p.numel() for block in model.trf_blocks for p in block.ff.parameters())

# print(att_params_count+ff_params_count)

# print(sum(p.numel() for p in model.trf_blocks.parameters()))


# deu diferente... é por causa das camadas de normalização
# print(sum(p.numel() for block in model.trf_blocks for p in list(block.norm1.parameters()) + list(block.norm2.parameters())))
# print(att_params_count+ff_params_count+sum(p.numel() for block in model.trf_blocks for p in list(block.norm1.parameters()) + list(block.norm2.parameters())))




for parametro in model.out_head.parameters():
    output_params_count += parametro.numel()



# output_params_count = sum(p.numel() for p in model.out_head.parameters())


print(f"Total de parâmetros da camada de embedding: {emb_params_count:,}")
print(f"Total de parâmetros dos módulos de atenção: {att_params_count:,}")
print(f"Total de parâmetros das camadas feed forward: {ff_params_count:,}")
print(f"Total de parâmetros da camada de output geracional: {output_params_count:,}")

# print(emb_params_count+att_params_count+ff_params_count+output_params_count+sum(p.numel() for block in model.trf_blocks for p in list(block.norm1.parameters()) + list(block.norm2.parameters()))+sum(p.numel() for p in model.final_norm.parameters()))

Total de parâmetros da camada de embedding: 39,383,808
Total de parâmetros dos módulos de atenção: 28,320,768
Total de parâmetros das camadas feed forward: 56,669,184
Total de parâmetros da camada de output geracional: 38,597,376


In [70]:
# aqui é rascunho do exemplo do livro. mais para eu pensar algo interessante

total_params = sum(p.numel() for p in model.parameters())

print(f"Total de parâmetros do modelo: {total_params:,}")



total_params_treinaveis = total_params - sum(p.numel() for p in model.out_head.parameters())

print(f"Total de parâmetros do modelo: {total_params_treinaveis:,}")

# das partes que vimos do modelo do transformer. São 163M de parametros, mas o citado é 124M. Livro fala q é pq no artigo original os pesquisadores aplicaram o compartilhamento de peso "self.out_head.weight = self.tok_emb.weight"




Total de parâmetros do modelo: 163,009,536
Total de parâmetros do modelo: 124,412,160


## 2 Dropout separado

Durante a criação do modelo GPT definimos apenas uma taxa de dropout, que é aplicada por todo o modelo. Altere o modelo para que existam 3 taxas de dropout distintas: uma na camada de embedding, outra na camada de shortcut e outra no módulo de atenção. Os parâmetros com as 3 taxas já foram definidos na configuração abaixo.

Além de fazer a alteração para incluir os novos dropouts, imprima (dentro da classe mesmo) a porcentagem de valores da matriz que ficaram zerados logo após passar pelo dropout, isso servirá para debugarmos se o dropout está funcionando conforme o esperado.

<small>DICA: conte os valores diferentes de zero, divida pelo total de elementos e subtraia 1 para contar a quantidade de zeros. <br>Ex: `(100 * (1 - torch.count_nonzero(x) / (x.numel())))`</small>

In [71]:
# ALTERE AS CLASSES MultiHeadAttention, TransformerBlock, GPTModel
# PARA ACRESCENTAR OS NOVOS DROPOUTS E IMPRIMIR A PORCENTAGEM DE PARÂMETROS
# ZERADOS DAS MATRIZES APÓS PASSAR PELO DROPOUT

# da ultima aula
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False, ID_do_bloco=0):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.ID_do_bloco = ID_do_bloco


        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        print(f"[Bloco de Transformer ID {self.ID_do_bloco}] {(100 * (1 - torch.count_nonzero(attn_weights) / (attn_weights.numel()))):.2f} % de valores da matriz do attention que ficaram zerados")

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec



class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))




class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)




class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift



class TransformerBlock(nn.Module):
    def __init__(self, cfg, ID_do_bloco = 0):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate_attn"],
            qkv_bias=cfg["qkv_bias"],
            ID_do_bloco=ID_do_bloco)
            
        self.ID_do_bloco = ID_do_bloco
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate_shortcut"]) # Já ta aqui tbm a parte do dropout no shortcut

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)

        print(f"[Bloco de Transformer ID {self.ID_do_bloco}] {(100 * (1 - torch.count_nonzero(x) / (x.numel()))):.2f} % de valores da matriz de shortcut que ficaram zerados")
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        print(f"[Bloco de Transformer ID {self.ID_do_bloco}] {(100 * (1 - torch.count_nonzero(x) / (x.numel()))):.2f} % de valores da matriz de shortcut que ficaram zerados")
        print("-"*100)
        x = x + shortcut  # Add the original input back

        return x




class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate_emb"]) # o exemplo do livro que me baseio já apresenta a parte do dropout embedding, só precisei mudar a "key" do dicionario que vou usar
        
        self.trf_blocks = nn.Sequential( # vai fazer esse trem de transformer 12 vezes no GPT2
            *[TransformerBlock(cfg, i) for i in range(cfg["n_layers"])]) # coloquei ID do bloco para deixar visualmente melhor o print (achei muito feio antes)
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x) # Basta realizar aqui o dropout_emb
        # como o professor pede para imprimir a % de zeros no dropout só fazer isso:
        print(f"[Começo do trem com embedding] {(100 * (1 - torch.count_nonzero(x) / (x.numel()))):.2f} % de valores da matriz de embedding que ficaram zerados") # literalmente o exemplo dele. Conto os zeros -> divido pelo total -> subtraio 1 -> multiplico por 100
        print("-"*100)

        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits





Dei uma tunada nos prints para facilitar a visualização. Por isso adicionei o "ID_do_bloco"

In [72]:
'''
Não precisa alterar ou completar nada nesta célula, apenas execute e veja as
porcentagens de elementos zerados de acordo com o print que você adicionou nas classes.
'''

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate_emb": 0.05,        # NOVO: dropout para camada de embedding
    "drop_rate_attn": 0.10,       # NOVO: dropout para o módulo de attention
    "drop_rate_shortcut": 0.15,   # NOVO: dropout para as shortcut connections
    "qkv_bias": False
}

model = GPTModel(GPT_CONFIG_124M)

inputs = torch.tensor(
  [[43, 15, 89],
   [55, 87, 66],
   [57, 85, 64],
   [22, 58, 33],
   [77, 25, 10],
   [15, 80, 55]]
)

output = model(inputs)

[Começo do trem com embedding] 5.01 % de valores da matriz de embedding que ficaram zerados
----------------------------------------------------------------------------------------------------
[Bloco de Transformer ID 0] 40.28 % de valores da matriz do attention que ficaram zerados
[Bloco de Transformer ID 0] 15.02 % de valores da matriz de shortcut que ficaram zerados
[Bloco de Transformer ID 0] 14.88 % de valores da matriz de shortcut que ficaram zerados
----------------------------------------------------------------------------------------------------
[Bloco de Transformer ID 1] 39.04 % de valores da matriz do attention que ficaram zerados
[Bloco de Transformer ID 1] 15.37 % de valores da matriz de shortcut que ficaram zerados
[Bloco de Transformer ID 1] 14.77 % de valores da matriz de shortcut que ficaram zerados
----------------------------------------------------------------------------------------------------
[Bloco de Transformer ID 2] 39.20 % de valores da matriz do attention